# 01 Data Audit - MetroPT-3 Dataset

This notebook starts the implementation stage. It loads the MetroPT-3 dataset, checks basic structure, confirms timestamp coverage, checks missing values and duplicate timestamps, and saves evidence tables for the dissertation.

Before running this notebook, place the raw MetroPT-3 dataset file inside `data/raw/`.


## 1. Import libraries and define paths

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

print('Python executable:', sys.executable)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)

# Notebook is expected to run from the notebooks/ folder
BASE_DIR = Path('..').resolve()
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
OUTPUT_TABLES_DIR = BASE_DIR / 'outputs' / 'tables'
OUTPUT_FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Project folder:', BASE_DIR)
print('Raw data folder:', RAW_DATA_DIR)


## 2. Find and load the dataset

The code below searches `data/raw/` for a CSV, TXT or Excel file. If there are multiple files, it uses the first one. Rename your raw file clearly if needed.


In [ ]:
candidate_files = []
for pattern in ['*.csv', '*.txt', '*.xlsx', '*.xls']:
    candidate_files.extend(RAW_DATA_DIR.glob(pattern))

if not candidate_files:
    raise FileNotFoundError(
        f'No dataset file found in {RAW_DATA_DIR}. Download MetroPT-3 and place the raw file inside data/raw/.'
    )

data_path = candidate_files[0]
print('Loading:', data_path.name)

if data_path.suffix.lower() in ['.xlsx', '.xls']:
    df = pd.read_excel(data_path)
else:
    df = pd.read_csv(data_path)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
df.head()


## 3. Inspect columns and data types

In [ ]:
columns_df = pd.DataFrame({
    'column_name': df.columns,
    'data_type': [str(dtype) for dtype in df.dtypes]
})
columns_df.to_csv(OUTPUT_TABLES_DIR / 'column_data_types.csv', index=False)
columns_df


## 4. Detect timestamp column and check date range

In [ ]:
possible_timestamp_cols = [
    col for col in df.columns
    if col.lower() in ['timestamp', 'time', 'datetime', 'date'] or 'time' in col.lower() or 'date' in col.lower()
]

print('Possible timestamp columns:', possible_timestamp_cols)

# Change this manually if the detected column is wrong.
timestamp_col = possible_timestamp_cols[0] if possible_timestamp_cols else 'timestamp'
print('Using timestamp column:', timestamp_col)

if timestamp_col not in df.columns:
    raise KeyError('Timestamp column not found. Please set timestamp_col manually to the correct column name.')

df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
df = df.sort_values(timestamp_col).reset_index(drop=True)

print('Start date:', df[timestamp_col].min())
print('End date:', df[timestamp_col].max())
print('Invalid timestamps:', df[timestamp_col].isna().sum())
print('Duplicate timestamps:', df.duplicated(subset=timestamp_col).sum())


## 5. Save dataset summary

In [ ]:
summary_df = pd.DataFrame({
    'rows': [df.shape[0]],
    'columns': [df.shape[1]],
    'start_date': [df[timestamp_col].min()],
    'end_date': [df[timestamp_col].max()],
    'invalid_timestamps': [df[timestamp_col].isna().sum()],
    'duplicate_timestamps': [df.duplicated(subset=timestamp_col).sum()]
})

summary_df.to_csv(OUTPUT_TABLES_DIR / 'dataset_summary.csv', index=False)
summary_df


## 6. Missing value audit

In [ ]:
missing_values = df.isnull().sum().reset_index()
missing_values.columns = ['column', 'missing_count']
missing_values['missing_percentage'] = (missing_values['missing_count'] / len(df)) * 100
missing_values = missing_values.sort_values('missing_count', ascending=False)

missing_values.to_csv(OUTPUT_TABLES_DIR / 'missing_values_summary.csv', index=False)
missing_values


## 7. Documented failure events

These documented failure intervals are used later for early-warning labels and event-level validation.


In [ ]:
failure_events = pd.DataFrame({
    'event_id': ['F1', 'F2', 'F3', 'F4'],
    'failure_type': ['Air leak', 'Air leak', 'Air leak', 'Air leak'],
    'failure_start': [
        '2020-04-18 00:00',
        '2020-05-29 23:30',
        '2020-06-05 10:00',
        '2020-07-15 14:30'
    ],
    'failure_end': [
        '2020-04-18 23:59',
        '2020-05-30 06:00',
        '2020-06-07 14:30',
        '2020-07-15 19:00'
    ]
})

failure_events['failure_start'] = pd.to_datetime(failure_events['failure_start'])
failure_events['failure_end'] = pd.to_datetime(failure_events['failure_end'])

failure_events['inside_dataset_range'] = (
    (failure_events['failure_start'] >= df[timestamp_col].min()) &
    (failure_events['failure_end'] <= df[timestamp_col].max())
)

failure_events.to_csv(OUTPUT_TABLES_DIR / 'documented_failure_events.csv', index=False)
failure_events


## 8. Separate numerical and non-numerical variables

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in df.columns if c not in numeric_cols]

variable_groups = pd.DataFrame({
    'group': ['numeric'] * len(numeric_cols) + ['non_numeric_or_datetime'] * len(non_numeric_cols),
    'column': numeric_cols + non_numeric_cols
})

variable_groups.to_csv(OUTPUT_TABLES_DIR / 'variable_groups.csv', index=False)
variable_groups


## 9. Audit outputs created

After running this notebook, check `outputs/tables/` for:

- `dataset_summary.csv`
- `missing_values_summary.csv`
- `column_data_types.csv`
- `documented_failure_events.csv`
- `variable_groups.csv`
